[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P03/P03-1_Fonctions.ipynb)

# Fonctions de calcul

Chapitre 3 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre bureau d'études vérifie chaque semaine des tiges, des poutres et des arbres avec les mêmes formules, recopiées d'une feuille de calcul à l'autre, et une seule erreur de recopie suffit à fausser un dimensionnement ? Vous voudriez écrire chaque formule une fois, la contrôler sur un cas connu, puis l'appliquer à cent configurations.

**Objectif terminal.** Ce chapitre montre comment ranger un calcul de résistance des matériaux dans une fonction, le tester, le réemployer, puis résoudre une équation sans solution toute faite.

**Objectif instrumental.**

- Écrire une fonction documentée qui reçoit des grandeurs et rend un résultat.
- La tester sur un cas connu et lui faire refuser un appel absurde.
- La réemployer, avec des arguments nommés, dans une boucle.
- Résoudre une équation par dichotomie avec une boucle `while`.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Aucun après une filière scientifique ; le chapitre 2 suffit.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 3.1 | Trou dans une plaque tendue | 10 min |
| 3.2 | Flèche d'une poutre sur deux appuis | 10 min |
| 3.3 | Diamètre d'un arbre en torsion (défi) | 20 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [ ]:
import io
import math

### Section 3.1 : une fonction s'écrit une fois et se teste.

In [ ]:
# notion: def
def SORTIE(ENTREE):
    """La forme la plus simple : une entrée, une sortie (programmation classique, règles écrites à la main)."""
    ENTREE = ENTREE.lower()

    if "capitale" in ENTREE:
        return "Paris"
    elif "monnaie" in ENTREE:
        return "Euro"
    else:
        return "Information non disponible"

In [ ]:
def MAPE(vrai, prevu):
    """Écart relatif moyen, en pourcentage, entre des valeurs prévues et des valeurs vraies."""
    somme = 0.0
    for v, p in zip(vrai, prevu):
        somme = somme + abs(v - p) / abs(v)
    return 100 * somme / len(vrai)

def contrainte_MPa(F_N, d_mm):
    """Contrainte normale dans une tige ronde, en MPa."""
    aire_mm2 = math.pi * d_mm**2 / 4
    return F_N / aire_mm2

print(SORTIE("Quelle est la capitale ?"), "|", SORTIE("Le poids ?"))
print(f"MAPE de (100, 200) prévu (110, 180) : {MAPE([100, 200], [110, 180]):.1f} %")
print(f"contrainte d'une tige de 10 mm sous 12 kN : {contrainte_MPa(12000.0, 10.0):.2f} MPa")
assert SORTIE("la monnaie") == "Euro"
assert abs(MAPE([100, 200], [110, 180]) - 10.0) < 1e-9, "MAPE attendue : 10 %"

In [ ]:
# notion: docstring
assert contrainte_MPa.__doc__ and "MPa" in contrainte_MPa.__doc__, "la docstring doit dire l'unité"
# notion: assert
assert abs(contrainte_MPa(12000.0, 10.0) - 152.79) < 0.005, "contrainte attendue : 152,79 MPa"

# notion: raise
def contrainte_gardee_MPa(F_N, d_mm):
    """Contrainte normale dans une tige ronde, en MPa ; un diamètre nul ou négatif n'a pas de sens."""
    if d_mm <= 0:
        raise ValueError("le diamètre doit être positif")
    return F_N / (math.pi * d_mm**2 / 4)

In [ ]:
# notion: try
try:
    contrainte_gardee_MPa(12000.0, 0.0)
    message = "aucune erreur"
except ValueError as erreur:
    message = str(erreur)

print("erreur interceptée :", message)
assert message == "le diamètre doit être positif"

In [ ]:
# notion: lambda
aire_mm2 = lambda d_mm: math.pi * d_mm**2 / 4
assert abs(aire_mm2(10.0) - 78.54) < 0.005, "aire attendue : 78,54 mm²"

## Exercice 3.1 : Trou dans une plaque tendue (10 min)

Une plaque de 50 mm de large et 5 mm d'épaisseur, percée d'un trou central de diamètre $d$, est tirée par 20 kN. Écrivez une fonction `kt_trou(d_mm, W_mm)` qui applique la formule de Heywood et rend `float("nan")` hors de son domaine. Calculez $K_t$ pour des trous de $5$, $10$, $20$ et $40$ mm, puis la contrainte nette et la contrainte maximale au bord d'un trou de 10 mm.

**Résultat attendu.** $K_t$ vaut 2,73, 2,51, 2,22 et 2,01 pour les quatre trous ; pour le trou de 10 mm, contrainte nette de 100,0 MPa et contrainte maximale de 251,2 MPa.

**Seconde voie.** À la main, la contrainte nette vaut $F/\bigl((W-d)\,e\bigr) = 20\,000/(40 \times 5)$, soit 100,0 MPa ; $K_t$ décroît quand le trou grossit et reste entre $2$ et $3$.

**Indication.** La fonction calcule le rapport $d/W$, rend `float("nan")` s'il n'est pas strictement compris entre zéro et un, puis rend la formule. Une boucle sur la liste des diamètres l'appelle quatre fois. La contrainte nette divise l'effort en newtons par la section restante $(W - d)\,e$ ; la contrainte maximale la multiplie par $K_t$.

Les contrôles de la cellule lisent les variables `kt_plaque`, `sigma_max_MPa` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(kt_plaque - 2.51) < 0.005, "Kt attendu : 2,51 pour d/W = 0,2"
assert abs(sigma_max_MPa - 251.2) < 0.05, "contrainte maximale attendue : 251,2 MPa"
print("Exercice 3.1 : contrôles passés")

### Section 3.2 : une même fonction s'emploie cent fois.

In [ ]:
# notion: reemploi
def securite(Re_MPa, F_N, d_mm):
    """Coefficient de sécurité : limite d'élasticité sur contrainte."""
    return Re_MPa / contrainte_MPa(F_N, d_mm)

for d in [8.0, 10.0, 12.0, 14.0]:
    print(f"d = {d:4.1f} mm : s = {securite(355.0, 12000.0, d):.2f}")

assert abs(securite(355.0, 12000.0, 10.0) - 2.32) < 0.005, "sécurité attendue : 2,32"

In [ ]:
# notion: kwargs
def fleche_defaut_mm(F_N, L_mm=1000.0, b_mm=20.0, h_mm=40.0, E_MPa=210000.0):
    """Flèche au centre d'une poutre sur deux appuis ; valeurs par défaut : poutre de l'exercice 3.2."""
    I_mm4 = b_mm * h_mm**3 / 12
    return F_N * L_mm**3 / (48 * E_MPa * I_mm4)

assert (
    securite(Re_MPa=355.0, d_mm=10.0, F_N=12000.0) == securite(355.0, 12000.0, 10.0)
), "ordre libre"
assert fleche_defaut_mm(2000.0) == fleche_defaut_mm(2000.0, h_mm=40.0), "valeur par défaut"
print(f"flèche avec les valeurs par défaut : {fleche_defaut_mm(2000.0):.2f} mm, "
      f"pour h = 60 mm : {fleche_defaut_mm(2000.0, h_mm=60.0):.2f} mm")

## Exercice 3.2 : Flèche d'une poutre sur deux appuis (10 min)

Une poutre en acier ($E = 210\,000$ MPa) repose sur deux appuis et porte une charge $F$ en son milieu. Sa flèche au centre vaut $F L^3 / (48\,E\,I)$, avec $I = b\,h^3/12$ pour une section rectangulaire. Écrivez la fonction, puis calculez la flèche pour $F = 2$ kN, $L = 1$ m et une section de $20 \times 40$ mm. Réemployez-la pour des hauteurs $h$ de $30$, $50$ et $60$ mm.

**Résultat attendu.** Flèche de 1,86 mm pour $h = 40$ mm ; 4,41 mm pour $h = 30$ mm, 0,95 mm pour $h = 50$ mm et 0,55 mm pour $h = 60$ mm.

**Seconde voie.** La flèche varie comme $1/h^3$ : doubler la hauteur la divise par huit, et le rapport des flèches de $30$ et de $60$ mm vaut huit.

**Indication.** Travaillez en newtons et en millimètres : $L = 1000$ mm, $E = 210\,000$ N/mm². Le moment quadratique se calcule dans la fonction. La flèche diminue vite avec la hauteur, puisque $I$ varie comme $h^3$.

Les contrôles de la cellule lisent la variable `f_mm` : gardez ce nom.

In [ ]:
# Votre code :
assert abs(f_mm - 1.86) < 0.005, "flèche attendue : 1,86 mm"
# Votre code :
print("Exercice 3.2 : contrôles passés")

## Exercice 3.3 : Diamètre d'un arbre en torsion (défi, 20 min)

Un arbre plein transmet un couple de $500$ N$\cdot$m. Son cisaillement maximal $\tau = 16\,T/(\pi d^3)$ ne doit pas dépasser $60$ MPa. Trouvez le plus petit diamètre admissible par dichotomie, entre $10$ et $100$ mm, en vous arrêtant quand l'encadrement fait moins de $5$ µ m, puis comparez avec la formule fermée.

**Résultat attendu.** Diamètre de 34,88 mm en 15 tours de boucle ; la formule fermée donne 34,8816 mm.

**Seconde voie.** La formule fermée $d = \bigl(16\,T/(\pi\,\tau)\bigr)^{1/3}$ donne 34,8816 mm, et le nombre de tours prévu par $\lceil \log_2(90/0{,}005) \rceil$ est celui de la boucle.

**Indication.** Écrivez une fonction du cisaillement en fonction du diamètre, avec le couple en N$\cdot$mm. À chaque tour, coupez l'encadrement au milieu : si le cisaillement dépasse $60$ MPa, l'arbre est trop fin et le milieu devient la borne basse ; sinon il devient la borne haute. Comptez les tours dans une variable, ou appelez la fonction `dichotomie` du chapitre avec l'écart $\tau - 60$.

Les contrôles de la cellule lisent les variables `d_dicho_mm`, `iterations` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(d_dicho_mm - 34.88) < 0.005, "diamètre attendu : 34,88 mm"
assert iterations == 15, "nombre d'itérations attendu : 15"
print("Exercice 3.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F03-1 : Tests intégrés (`doctest.testmod`)

In [ ]:
import doctest

def sigma_MPa(p_MPa, r_mm, e_mm):
    """
    >>> round(sigma_MPa(2.5, 800.0, 15.0), 1)
    133.3
    """
    return p_MPa * r_mm / e_mm

res = doctest.testmod()
c = sigma_MPa(2.5, 800.0, 15.0)
print(f"Essai détecté : {res.attempted >= 1}, contrainte : {c:.1f} MPa".replace(".", ","))
print(res.failed == 0)